# 04 Salinity analysis

One question (report Section 4.3): do the HH/VV ratio or the CPD change with salinity once everything else that changes them is accounted for? The expected signal is tiny (notebook 03: about 0.1 dB and 0.03 deg across the whole Amazon plume), so most of this notebook is about making sure an apparent signal isn't caused by something else.

1. **Instrument behaviour.** Repeat-pass stability, the CPD trend across each swath, and ascending vs descending.
2. **Matching Biomass to SMOS** on the SMOS 0.25 deg grid, keeping only cells seen by SMOS within 60 min, at least 60 % covered by Biomass, and more than 40 km from land.
3. **Within each pass.** Pass means and the incidence angle trend are removed, since calibration and geometry differ between passes.
4. **The controlled regression.** Predict SMOS salinity from the radar, then check where the skill actually comes from by refitting with each group of inputs on its own and after removing position, wind and swell.
5. **A radar-only index** built with no salinity input, and how much of its correlation with SMOS survives the same controls.

It reads the `.npz` files from notebook 02. Downloading SMOS, SMAP, ERA5 and the wave model is in notebook 01.

In [ ]:
# ---- the only cell that needs editing ----
from pathlib import Path

NPZ_DIR = Path(r"C:\Users\Orlaith.Doyle\biomass_processed")
NPZ_GLOB = "T02[018]_*.npz"          # which frames to use, here the three Amazon tracks
OUT = Path(r"C:\Users\Orlaith.Doyle\biomass_salinity")

SMOS_ASC = Path(r"C:\Users\Orlaith.Doyle\refs\smos_asc")    # folders of CMEMS SMOS L3 daily files
SMOS_DES = Path(r"C:\Users\Orlaith.Doyle\refs\smos_des")
SMAP_DIR = Path(r"C:\Users\Orlaith.Doyle\refs\smap")        # RSS v6 8 day, only for reference. None to skip
ERA5 = Path(r"C:\Users\Orlaith.Doyle\refs\era5_wind_amazon.nc")
WAVES = Path(r"C:\Users\Orlaith.Doyle\refs\cmems_wav_amazon.nc")
SMOS_TIME_VAR = None                  # None = find it automatically

MAX_DT_MIN = 60       # SMOS seen within this many minutes of Biomass
MIN_COVER = 0.6       # fraction of the SMOS cell covered by Biomass ocean
MIN_COAST_KM = 40     # SMOS is contaminated near the coast
MIN_SSS_RANGE = 2.0   # psu, a pass needs at least this much salinity contrast for the regression

# repeat pass stability check (Guiana shelf)
REPEAT_GLOB = "T036_F007_*.npz"

In [ ]:
import sys
sys.path.append(str(Path.cwd()))
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import ndimage
import bm_tools as bm

OUT.mkdir(parents=True, exist_ok=True)

def save(fig, what):
    fig.savefig(OUT / f"{what}.png", dpi=200, bbox_inches="tight")
    print("saved", what + ".png")

frames = []
for f in sorted(NPZ_DIR.glob(NPZ_GLOB)):
    d = {k: (v.item() if v.ndim == 0 else v) for k, v in np.load(f, allow_pickle=True).items()}   # 0-d arrays back to plain values
    d["t0"] = np.datetime64(str(d["t0"]))
    d["pass"] = f"T{int(d['track']):03d}_{str(d['t0'])[:10]}_{str(d['orbit_pass'])[:3]}"
    frames.append(d)
print(len(frames), "frames,", len({d["pass"] for d in frames}), "passes")

## 1. Instrument behaviour

**Repeat passes.** The same frame imaged on different days, where salinity barely changes, shows how much the instrument itself wanders. On the six Guiana shelf passes the frame mean CPD moved by about 0.5 deg and the backscatter by about 1.4 dB. That's far bigger than the salinity signal, and it's why everything after this works within a pass.

**Across the swath.** The CPD rises steadily from near to far range, 2.2 to 3.6 deg depending on swath and orbit direction.

**Ascending vs descending.** The mean CPD also differs between dawn and dusk passes and changes sign on S3. The physics doesn't predict that, so it's noted as a calibration item. In a dawn-dusk orbit the pass direction and local time (and so the state of the ionosphere) can't be separated.

In [ ]:
rep = []
for f in sorted(NPZ_DIR.glob(REPEAT_GLOB)):
    d = np.load(f, allow_pickle=True)
    rep.append({"date": str(d["t0"])[:10], "pass": str(d["orbit_pass"]),
                "cpd": np.degrees(np.angle(np.nansum(d["hhvv"]))),
                "coh": np.abs(np.nansum(d["hhvv"])) / np.sqrt(np.nansum(d["hh"]) * np.nansum(d["vv"])),
                "vv_db": 10 * np.log10(np.nanmean(d["s0vv"]))})
rep = pd.DataFrame(rep)
if len(rep):
    print(rep.round(3).to_string(index=False))
    print("between passes: CPD range %.2f deg, VV range %.2f dB" % (np.ptp(rep.cpd), np.ptp(rep.vv_db)))

# CPD trend across the swath and the mean CPD, per swath and orbit direction
beh = []
for d in frames:
    g = np.isfinite(d["cpd"])
    if g.sum() < 100:
        continue
    inc, z = d["inc"][g], d["hhvv"][g]
    b = np.linspace(inc.min(), inc.max(), 11)
    mid = (b[:-1] + b[1:]) / 2
    cb = np.array([np.degrees(np.angle(np.sum(z[(inc >= lo) & (inc < hi)]))) if ((inc >= lo) & (inc < hi)).any() else np.nan
                   for lo, hi in zip(b[:-1], b[1:])])
    ok = np.isfinite(cb)
    slope = np.polyfit(mid[ok], cb[ok], 1)[0] if ok.sum() > 3 else np.nan
    beh.append({"swath": str(d["swath"]), "orbit": str(d["orbit_pass"])[:3], "pass": d["pass"],
                "mean_cpd": np.degrees(np.angle(np.sum(z))), "cpd_across_swath": slope * np.ptp(inc)})
beh = pd.DataFrame(beh)
print(beh.groupby(["swath", "orbit"])[["mean_cpd", "cpd_across_swath"]].agg(["mean", "count"]).round(2))

## 2. Matching Biomass to SMOS

Biomass and SMOS are compared on the SMOS 0.25 deg grid. For every SMOS cell under a pass, the multilooked Biomass second order products from all frames of that pass are summed over the ocean cells inside it, and the CPD and HH/VV ratio are worked out from those sums (so it's never averaging dB or angles).

A cell is kept only if:

- SMOS saw it within `MAX_DT_MIN` of Biomass, from SMOS's own per pixel acquisition time. Both fly dawn-dusk orbits, so ascending Biomass goes with ascending SMOS and descending with descending.
- Biomass ocean covers at least `MIN_COVER` of it.
- it's more than `MIN_COAST_KM` from land.

ERA5 wind and the wave model swell and wind sea heights are sampled at the same cells, at the closest time.

In [ ]:
def open_smos(folder, day):
    # the file that holds this day (one file per day or one long file both work, no dask needed)
    for f in sorted(Path(folder).glob("*.nc")):
        ds = xr.open_dataset(f)
        if "time" not in ds.dims or np.datetime64(day, "D") in ds.time.values.astype("datetime64[D]"):
            break
    else:
        raise FileNotFoundError(f"no SMOS file in {folder} covers {day}")
    if "time" in ds.dims:
        ds = ds.sel(time=np.datetime64(day, "D"), method="nearest")
    sss = ds["Sea_Surface_Salinity"].squeeze()
    tv = SMOS_TIME_VAR or next((v for v in ds.data_vars if ("time" in v.lower() or "acq" in v.lower())
                                and ds[v].ndim >= 2), None)
    if tv is None:
        print("  !! no per pixel time in the SMOS file, the 60 min check is OFF. variables:", list(ds.data_vars))
        tt = None
    else:
        tt = ds[tv].squeeze()
        if np.issubdtype(tt.dtype, np.timedelta64):
            tt = np.datetime64(day, "D") + tt.values
        elif np.issubdtype(tt.dtype, np.datetime64):
            tt = tt.values
        else:
            u = tt.attrs.get("units", "").lower()
            f = 3600 if "hour" in u else 60 if "minute" in u else 1 if "second" in u else None
            if f is None:
                print(f"  !! don't know the units of {tv} ({u}), set SMOS_TIME_VAR / fix this. time check OFF")
                tt = None
            else:
                tt = np.datetime64(day, "D") + (tt.values * f * 1e3).astype("timedelta64[ms]")
    return sss, tt


def coast_km(lat, lon, pad=1.5, res=0.02):
    """distance to the nearest land in km, from a local land raster (global_land_mask)"""
    from global_land_mask import globe
    la = np.arange(np.nanmin(lat) - pad, np.nanmax(lat) + pad, res)
    lo = np.arange(np.nanmin(lon) - pad, np.nanmax(lon) + pad, res)
    LO, LA = np.meshgrid((lo + 180) % 360 - 180, la)
    land = globe.is_land(LA, LO)
    if not land.any():
        return np.full(np.shape(lat), 999.0)
    dk = ndimage.distance_transform_edt(~land, sampling=(res * 111.2, res * 111.2 * np.cos(np.radians(np.mean(la)))))
    return bm.sample_grid(la, lo, dk, lat, lon)


era = xr.open_dataset(ERA5) if ERA5 and Path(ERA5).exists() else None
wav = xr.open_dataset(WAVES) if WAVES and Path(WAVES).exists() else None

rows = []
passes = sorted({d["pass"] for d in frames})
for ip, ps in enumerate(passes):
    fr = [d for d in frames if d["pass"] == ps]
    t_pass = fr[0]["t0"]
    sss, tt = open_smos(SMOS_ASC if fr[0]["orbit_pass"].startswith("A") else SMOS_DES, str(t_pass)[:10])
    sla, slo = sss["latitude"].values, sss["longitude"].values
    dla, dlo = np.diff(sla).mean(), np.diff(slo).mean()
    acc = {}
    for d in fr:
        g = np.isfinite(d["hh"]) & np.isfinite(d["vv"])
        ia = np.round((d["lat"][g] - sla[0]) / dla).astype(int)
        ja = np.round((d["lon"][g] - slo[0]) / dlo).astype(int)
        cell_area = float(d["naz"]) * float(d["d_az"]) * float(d["nrg"]) * float(d["d_rg"]) * d["wfrac"][g]
        for key, vals in [("hh", d["hh"][g]), ("vv", d["vv"][g]), ("hhvv", d["hhvv"][g]), ("inc", d["inc"][g]),
                          ("lat", d["lat"][g]), ("lon", d["lon"][g]), ("area", cell_area), ("n", np.ones(g.sum()))]:
            df = pd.DataFrame({"i": ia, "j": ja, "v": vals})
            s = df.groupby(["i", "j"])["v"].sum()
            acc[key] = acc[key].add(s, fill_value=0) if key in acc else s
    cells = pd.DataFrame(acc).reset_index()
    cells = cells[(cells.i >= 0) & (cells.i < len(sla)) & (cells.j >= 0) & (cells.j < len(slo))]
    cells["pass"], cells["track"], cells["orbit"] = ps, int(fr[0]["track"]), str(fr[0]["orbit_pass"])[:3]
    cells["swath"], cells["t_bio"] = str(fr[0]["swath"]), t_pass
    cells["inc"], cells["lat"], cells["lon"] = cells.inc / cells.n, cells.lat / cells.n, cells.lon / cells.n
    cells["ratio_db"] = 10 * np.log10(cells.hh / cells.vv)
    cells["cpd"] = np.degrees(np.angle(cells.hhvv.astype(complex)))
    cells["coh"] = np.abs(cells.hhvv.astype(complex)) / np.sqrt(cells.hh * cells.vv)
    smos_area = (dla * 111.2e3) * (dlo * 111.2e3 * np.cos(np.radians(sla[cells.i.values])))
    cells["cover"] = cells.area / np.abs(smos_area)
    cells["sss"] = sss.values[cells.i.values, cells.j.values]
    cells["dt_min"] = (np.abs(tt[cells.i.values, cells.j.values] - np.datetime64(t_pass)) / np.timedelta64(1, "m")
                       if tt is not None else np.nan)
    cells["coast_km"] = coast_km(cells.lat.values, cells.lon.values)
    if era is not None:
        e = bm.nearest_time(era, t_pass)
        cells["wind"] = np.hypot(bm.interp_ll(e["u10"], cells.lat.values, cells.lon.values),
                                 bm.interp_ll(e["v10"], cells.lat.values, cells.lon.values))
    if wav is not None:
        w = bm.nearest_time(wav, t_pass)
        cells["swell"] = bm.interp_ll(w["VHM0_SW1"], cells.lat.values, cells.lon.values)
        cells["windsea"] = bm.interp_ll(w["VHM0_WW"], cells.lat.values, cells.lon.values)
    rows.append(cells)
    print(f"  pass {ip + 1}/{len(passes)} {ps}: {len(cells)} SMOS cells under it")

allc = pd.concat(rows, ignore_index=True)
keep = (allc.cover >= MIN_COVER) & (allc.coast_km > MIN_COAST_KM) & np.isfinite(allc.sss)
if allc.dt_min.notna().any():
    keep &= allc.dt_min <= MAX_DT_MIN
m = allc[keep].copy()
print(f"kept {len(m)} of {len(allc)} cells (cover, coast, SMOS and time checks)")
m.drop(columns=["hhvv"]).to_csv(OUT / "matched_cells.csv", index=False)

In [ ]:
# optional: SMAP 8 day mean at the same cells, as a second radiometer to look at. not used in the regression
if SMAP_DIR and Path(SMAP_DIR).exists():
    for ps, g in m.groupby("pass"):
        day = np.datetime64(str(g.t_bio.iloc[0])[:10], "D")
        fs = sorted(Path(SMAP_DIR).glob(f"*{str(day).replace('-', '')}*.nc")) or sorted(Path(SMAP_DIR).glob("*.nc"))
        if not fs:
            continue
        s = xr.open_dataset(fs[0])
        v = "sss_smap" if "sss_smap" in s else next(iter(s.data_vars))
        da = s[v].squeeze()
        if "time" in da.dims:
            da = da.sel(time=day, method="nearest")
        m.loc[g.index, "sss_smap"] = bm.interp_ll(da, g.lat.values, g.lon.values)
    if "sss_smap" in m:
        print("SMOS vs SMAP over the matched cells: r = %.2f" % m[["sss", "sss_smap"]].corr().iloc[0, 1])

## 3. Within each pass

Calibration offsets and viewing geometry differ from pass to pass, so values from different passes can't be compared directly. Each pass is treated on its own: its mean is removed and only the variation inside the pass is compared with salinity. The dependence on incidence angle across the swath is taken out too, with a straight line fit per pass, because both observables change with incidence far more than with salinity.

The scatter left inside a pass is the noise any salinity signal has to beat. It came out at about 0.2 deg in CPD and 0.4 dB in HH/VV between SMOS sized cells, against an expected plume signal of 0.03 deg and 0.1 dB.

In [ ]:
def within(g, cols):
    g = g.copy()
    for c in cols:
        if c not in g:
            continue
        y = g[c].values.astype(float)
        if c in ("ratio_db", "cpd"):
            A = np.column_stack([np.ones(len(g)), g.inc.values])     # remove mean and incidence trend
        else:
            A = np.ones((len(g), 1))                                 # just the pass mean
        ok = np.isfinite(y)
        coef = np.linalg.lstsq(A[ok], y[ok], rcond=None)[0]
        g[c + "_a"] = y - A @ coef
    return g

cols = ["ratio_db", "cpd", "sss", "wind", "swell", "windsea", "lat"]
m = pd.concat([within(g, cols) for _, g in m.groupby("pass")])
m["sss_range"] = m.groupby("pass").sss.transform(lambda s: np.nanpercentile(s, 95) - np.nanpercentile(s, 5))

scat = m.groupby("pass")[["cpd_a", "ratio_db_a"]].std()
print("within pass scatter (median over passes): CPD %.2f deg, HH/VV %.2f dB"
      % (scat.cpd_a.median(), scat.ratio_db_a.median()))

R = m[m.sss_range >= MIN_SSS_RANGE].dropna(subset=["cpd_a", "ratio_db_a", "sss_a"]).copy()
print(f"{R['pass'].nunique()} passes with at least {MIN_SSS_RANGE} psu of contrast, {len(R)} cells")

# inside a plume pass, swell and salinity both follow distance offshore
for ps, g in R.groupby("pass"):
    if "swell" in g and g.swell.notna().sum() > 3:
        print(f"  {ps}: r(swell, distance offshore) = {np.corrcoef(g.swell, g.coast_km)[0, 1]:+.2f}, "
              f"r(SMOS, distance offshore) = {np.corrcoef(g.sss, g.coast_km)[0, 1]:+.2f}")

## 4. The controlled regression

A linear regression predicts the within-pass SMOS salinity anomaly. It's cross validated by leaving one pass out at a time: fit on all the other passes, predict the one left out. Skill is measured against simply using the pass mean (an anomaly of zero), so skill 0 means no better than the pass mean.

The question is where any skill comes from, so it's refit with each group of inputs on its own:

- **radar**: HH/VV and CPD anomalies
- **ERA5 and waves**: wind, swell and wind sea height
- **position**: latitude within the pass, standing in for along track position

and once more with the radar after latitude has been regressed out of everything inside each pass.

This is a conservative test. The controls vary smoothly along track, so they can remove part of a real salinity gradient too. A signal that doesn't survive them isn't proof the radar can't see salinity, only that here it can't be separated from sea state and position. Repeat passes over a fixed location are the cleaner way to get at it.

In [ ]:
def partial_out(df, ycols, xcols):
    """inside each pass, regress ycols on xcols and keep the residuals"""
    out = df.copy()
    for ps, g in df.groupby("pass"):
        X = np.column_stack([np.ones(len(g))] + [g[c].values for c in xcols])
        for y in ycols:
            coef = np.linalg.lstsq(X, g[y].values, rcond=None)[0]
            out.loc[g.index, y] = g[y].values - X @ coef
    return out


def lopo(df, xcols, y="sss_a"):
    df = df.dropna(subset=xcols + [y])
    pred = pd.Series(np.nan, index=df.index)
    for ps in df["pass"].unique():
        tr, te = df["pass"] != ps, df["pass"] == ps
        X = np.column_stack([np.ones(tr.sum())] + [df.loc[tr, c] for c in xcols])
        coef = np.linalg.lstsq(X, df.loc[tr, y], rcond=None)[0]
        pred[te] = np.column_stack([np.ones(te.sum())] + [df.loc[te, c] for c in xcols]) @ coef
    sse, sse0 = ((df[y] - pred) ** 2).sum(), (df[y] ** 2).sum()
    better = sum(((g[y] - pred[g.index]) ** 2).sum() < (g[y] ** 2).sum() for _, g in df.groupby("pass"))
    return {"skill": 1 - sse / sse0, "r": np.corrcoef(pred, df[y])[0, 1],
            "rmse": np.sqrt(sse / len(df)), "rmse_passmean": np.sqrt(sse0 / len(df)),
            "passes improved": f"{better}/{df['pass'].nunique()}"}


radar = ["ratio_db_a", "cpd_a"]
env = [c for c in ["wind_a", "swell_a", "windsea_a"] if c in R and R[c].notna().any()]
pos = ["lat_a"]

tests = {"All": radar + env + pos, "Radar only": radar, "ERA5 wind and waves only": env, "Latitude only": pos}
res = {k: lopo(R, v) for k, v in tests.items() if v}
Rl = partial_out(R, radar + ["sss_a"], ["lat_a"])
res["Radar, latitude removed"] = lopo(Rl, radar)
skill = pd.DataFrame(res).T
skill.to_csv(OUT / "regression_skill.csv")
print(skill.round(2).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(skill.index[::-1], skill.skill.astype(float)[::-1], color="0.4")
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("skill vs pass mean (leave one pass out)")
ax.set_title(f"{R['pass'].nunique()} passes, {len(R)} SMOS cells")
save(fig, "regression_skill")
plt.show()

## 5. A radar-only index

The same picture from the other direction, with no salinity input at all. The two radar anomalies are standardised and combined into their first principal component, oriented so that a higher index is what theory predicts for fresher water (HH/VV and CPD both go up as salinity drops). It should then correlate negatively with SMOS.

Then the same correlation after position, wind and swell are removed inside each pass. If it drops to around zero, the radar is following the sea state and position, not the salt.

In [ ]:
Z = (R[radar] - R[radar].mean()) / R[radar].std()
w, V = np.linalg.eigh(np.cov(Z.T))
pc = V[:, -1] * np.sign(V[0, -1])               # ratio loading positive
R["index"] = Z.values @ pc
print("index loadings (HH/VV, CPD):", pc.round(2))

r_all = np.corrcoef(R["index"], R.sss_a)[0, 1]
per = pd.Series({ps: np.corrcoef(g["index"], g.sss_a)[0, 1] for ps, g in R.groupby("pass")})
Rc = partial_out(R, ["index", "sss_a"], pos + env)
r_ctrl = np.corrcoef(Rc["index"], Rc.sss_a)[0, 1]
print("index vs SMOS: r = %.2f overall, per pass %.2f to %.2f" % (r_all, per.min(), per.max()))
print("after removing position, wind and swell: r = %.2f" % r_ctrl)

# how big a correlation chance alone gives here: shuffle the index inside each pass
rng = np.random.default_rng(0)
null = []
for _ in range(500):
    sh = Rc.groupby("pass")["index"].transform(lambda s: rng.permutation(s.values))
    null.append(np.corrcoef(sh, Rc.sss_a)[0, 1])
print("chance alone: 95 %% of shuffles within +-%.2f" % np.percentile(np.abs(null), 95))

n = R["pass"].nunique()
ncol = min(4, n)
fig, axs = plt.subplots(int(np.ceil(n / ncol)), ncol, figsize=(3.6 * ncol, 3.2 * np.ceil(n / ncol)),
                        squeeze=False, layout="constrained")
for a, (ps, g) in zip(axs.ravel(), R.groupby("pass")):
    a.scatter(g.sss, g["index"], s=10, c=g.coast_km, cmap="viridis")
    a.set_title(f"{ps}\nr = {np.corrcoef(g['index'], g.sss_a)[0, 1]:+.2f}", fontsize=8)
    a.set_xlabel("SMOS SSS (psu)", fontsize=8); a.set_ylabel("radar index", fontsize=8)
for a in axs.ravel()[n:]:
    a.axis("off")
fig.suptitle("radar-only index vs SMOS per pass (colour = km from coast)")
save(fig, "radar_index_per_pass")
plt.show()